<a href="https://colab.research.google.com/github/leorasdsouza/Hybrid-Cipher/blob/main/HybridCipher.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
!pip install pycryptodome

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.3/2.3 MB 26.7 MB/s eta 0:00:00


In [ ]:
from Crypto.Cipher import AES
from Crypto.Util.Padding import pad, unpad
from Crypto.Random import get_random_bytes
import random

# Generate a 128-bit AES key
AES_KEY = get_random_bytes(16)  # 16 bytes = 128 bits

# Generate a valid transposition key dynamically
def generate_transposition_key(num_blocks):
    key = list(range(num_blocks))
    random.shuffle(key)  # Shuffle to create a random permutation
    return key

# Substitution: AES Encryption
def aes_encrypt(plaintext, key):
    cipher = AES.new(key, AES.MODE_CBC)
    ct_bytes = cipher.encrypt(pad(plaintext.encode(), AES.block_size))
    return cipher.iv + ct_bytes  # Return IV + ciphertext

def aes_decrypt(ciphertext, key):
    iv = ciphertext[:AES.block_size]  # Extract IV
    ct = ciphertext[AES.block_size:]  # Extract ciphertext
    cipher = AES.new(key, AES.MODE_CBC, iv)
    pt = unpad(cipher.decrypt(ct), AES.block_size)
    return pt.decode()

# Transposition: Block-level Permutation (with padding)
def transpose_blocks(data, block_size, key):
    while len(data) % block_size != 0:
        data += b' '  # Padding to complete last block

    blocks = [data[i:i+block_size] for i in range(0, len(data), block_size)]
    num_blocks = len(blocks)

    # Ensure key matches the number of blocks
    if len(key) != num_blocks:
        key = list(range(num_blocks))  # Default sequential order

    transposed_blocks = [blocks[i] for i in key]
    return b''.join(transposed_blocks)

# Hybrid Encryption
def hybrid_encrypt(plaintext, aes_key, transposition_key):
    aes_ciphertext = aes_encrypt(plaintext, aes_key)
    block_size = 16
    transposed_ciphertext = transpose_blocks(aes_ciphertext, block_size, transposition_key)
    return transposed_ciphertext

# Hybrid Decryption
def hybrid_decrypt(ciphertext, aes_key, transposition_key):
    block_size = 16
    num_blocks = len(ciphertext) // block_size

    reverse_key = sorted(range(len(transposition_key)), key=lambda x: transposition_key[x])
    untransposed_ciphertext = transpose_blocks(ciphertext, block_size, reverse_key)
    plaintext = aes_decrypt(untransposed_ciphertext, aes_key)
    return plaintext.strip()

# Example usage
plaintext = input("Enter text: ")
num_blocks = len(plaintext) // 16 + 1  # Calculate required blocks
TRANSPOSITION_KEY = generate_transposition_key(num_blocks)

print("Plaintext:", plaintext)

ciphertext = hybrid_encrypt(plaintext, AES_KEY, TRANSPOSITION_KEY)
print("Ciphertext (hex):", ciphertext.hex())

decrypted_text = hybrid_decrypt(ciphertext, AES_KEY, TRANSPOSITION_KEY)
print("Decrypted Text:", decrypted_text)


Enter text: This is a secret message
Plaintext: This is a secret message
Ciphertext (hex): 5cc052ade1e05cb059329e6538059d99435a054246f5968e71f16cbc261135ac853259ceb379582122e136821cb172fe
Decrypted Text: This is a secret message
